In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import numpy as np
from sklearn.impute import KNNImputer
import os 

In [ ]:
test_df = pd.read_csv("../../datasets/processados/tratados/10-06-2023/bbr/tratado bbr esmond data mg-rs 06-10-2023.csv")

In [ ]:
def linear_interpolation (df): 
    df_copy = df.copy()

    df_copy['Data'] = pd.to_datetime(df_copy['Data'], dayfirst=True)

    df.set_index('Data', inplace=True)

    df['Vazao'] = df['Vazao'].replace(-1, np.nan).interpolate(method='linear', limit_direction='backward', order=1, replace=True)

    df = df.ffill()

    return df

df = linear_interpolation(test_df)
print(df['Vazao'].isnull().values.any())
df

In [ ]:
def knn (df):
    df_copy = df.copy()

    df_copy['Vazao'] = df_copy['Vazao'].replace(-1.0, np.nan)
    
    vazao_array = df_copy['Vazao'].values.reshape(-1, 1)
    
    imputer = KNNImputer(n_neighbors=1) #No experimento foi usado k = 1? k=5?
    
    df_copy['Vazao'] = imputer.fit_transform(vazao_array)
    
    return df_copy

df = knn(test_df)
print(df['Vazao'].isnull().values.any())
df.head(50)

In [ ]:
def moving_average (df):
    df_original = df
    df_copy = df["Vazao"]
    
    df_copy = df_copy.replace(-1, np.nan).fillna(df_copy.rolling(30,min_periods=1).mean())
    df_copy = df_copy.replace(-1, np.nan)
    df_copy = df_copy.bfill()
    df_original["Vazao"] = df_copy

    return df_original

df = moving_average(test_df)
print(df['Vazao'].isnull().values.any())
df.head(50)

In [ ]:
def moving_median (df):
    df_original = df
    df_copy = df["Vazao"]
    
    # Preenchendo os valores -1 com NaN e realizando 
    # Realizando o cálculo de rolling mean quantas vezes for necessário para preencher todos os dados   (como foi que eu fiz isso?)
    df_copy = df_copy.replace(-1, np.nan)
    df_copy = df_copy.fillna(df_copy.rolling(6,min_periods=1).median())
    df_copy = df_copy.bfill()
    df_original["Vazao"] = df_copy
    return df_original

df = moving_median(test_df)
print(df['Vazao'].isnull().values.any())
df.head(50)

In [ ]:
def all_imputation_methods(routes):
    protocols = ["bbr", "cubic"]
    for protocol in protocols:
        for route in routes:
            arq = f'../../datasets/processados/tratados/10-06-2023/{protocol}/tratado {protocol} esmond data {route} 06-10-2023.csv'
            name = f'tratado {protocol} esmond data {route} 06-10-2023'
            diretory_for_saving = f'../../datasets/processados/tratados_imputados/vazao/{protocol}/{route}/{name}'
            df = pd.read_csv(arq)
            linear_interp_df = linear_interpolation(df)
            linear_interp_df.to_csv(f'{diretory_for_saving}_interpolacao-linear.csv')

            knn_df = knn(df)
            knn_df.to_csv(f'{diretory_for_saving}_knn.csv')

            moving_average_df = moving_average(df)
            moving_average_df.to_csv(f'{diretory_for_saving}_media-movel.csv')

            moving_median_df = moving_median(df)
            moving_median_df.to_csv(f'{diretory_for_saving}_mediana-movel.csv')